# 6) Redispatch and N-1 security — Exercises

**Goals:** build the LODFs and verify them, screen every outage against every line, price short-term emergency ratings, and find which reinforcement N-1 security values most.

The setup cell holds the lecture's network, `PTDF`, `flows(dispatch, ptdf_matrix)`, lesson 4's N-0 dispatch `N0_DISPATCH`, and `secure_dispatch(lodf, limits, post_limits, security)` — the lecture's security-constrained dispatch with a limit per line before and after an outage, which needs **your** LODF matrix. Power in MW, costs in €/h.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")
logging.getLogger("linopy").setLevel(logging.ERROR)

# ---- the lecture's network, plants and PTDFs ----
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
LINES.index = [f"{a}-{b}" for a, b in zip(LINES.bus0, LINES.bus1)]
X_KM, LIMIT = 0.32, 400.0
LOAD = pd.Series({"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250, "Velebit": 0})[BUSES]
GENS = pd.DataFrame([("wind Velebit", "Velebit", 600, 0.0), ("hydro Split", "Split", 700, 30.0),
                     ("gas Rijeka", "Rijeka", 600, 90.0), ("coal Osijek", "Osijek", 400, 80.0),
                     ("import Zagreb", "Zagreb", 800, 130.0)],
                    columns=["name", "bus", "p_max_MW", "cost_EUR_per_MWh"]).set_index("name")


def ptdf(lines=LINES, reference="Zagreb"):
    idx = {b: i for i, b in enumerate(BUSES)}
    b = 1 / (X_KM * lines.km.to_numpy() / (400 ** 2 / 100))
    A = np.zeros((len(lines), len(BUSES)))
    A[range(len(lines)), lines.bus0.map(idx)] = 1
    A[range(len(lines)), lines.bus1.map(idx)] = -1
    B = A.T @ np.diag(b) @ A
    keep = [i for i, name in enumerate(BUSES) if name != reference]
    B_inv = np.zeros_like(B)
    B_inv[np.ix_(keep, keep)] = np.linalg.inv(B[np.ix_(keep, keep)])
    return pd.DataFrame(np.diag(b) @ A @ B_inv, index=lines.index, columns=BUSES)


PTDF = ptdf()
G = np.zeros((len(BUSES), len(GENS)))
for k, bus in enumerate(GENS.bus):
    G[BUSES.index(bus), k] = 1
GI = pd.Index(GENS.index, name="gen")
COST = xr.DataArray(GENS.cost_EUR_per_MWh.to_numpy(), coords=[GI])


def flows(dispatch, ptdf_matrix=PTDF):
    return pd.Series(ptdf_matrix.to_numpy() @ (G @ dispatch[GENS.index].to_numpy() - LOAD.to_numpy()), index=ptdf_matrix.index)


def secure_dispatch(lodf, limits=None, post_limits=None, security=True):
    """The lecture's (security-constrained) dispatch with a limit per line before (limits) and after an outage
    (post_limits); both default to 400 MW. Pass your own LODF matrix."""
    limits = pd.Series(LIMIT, index=LINES.index) if limits is None else limits
    post_limits = limits if post_limits is None else post_limits
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[GI]), name="p")
    m.add_constraints(p.sum() == LOAD.sum(), name="balance")
    M, c0 = PTDF.to_numpy() @ G, PTDF.to_numpy() @ LOAD.to_numpy()
    f = {l: (p * xr.DataArray(M[i], coords=[GI])).sum() - float(c0[i]) for i, l in enumerate(LINES.index)}
    for l in LINES.index:
        m.add_constraints(f[l] <= limits[l], name=f"upper {l}")
        m.add_constraints(f[l] >= -limits[l], name=f"lower {l}")
    if security:
        for k in LINES.index:
            for l in LINES.index.drop(k):
                post = f[l] + lodf.loc[l, k] * f[k]
                m.add_constraints(post <= post_limits[l], name=f"upper {l} | {k} out")
                m.add_constraints(post >= -post_limits[l], name=f"lower {l} | {k} out")
    m.add_objective((p * COST).sum())
    m.solve(solver_name="highs", output_flag=False)
    return {"cost": float(m.objective.value), "dispatch": p.solution.to_pandas()}


N0_DISPATCH = pd.Series({"wind Velebit": 485.0, "hydro Split": 700.0, "gas Rijeka": 300.0, "coal Osijek": 400.0,
                         "import Zagreb": 15.050505})            # lesson 4's N-0 dispatch, 81,956.3 EUR/h

### Warm-ups

**1. The LODFs**

Write `lodf()` returning the LODF matrix (rows: monitored line ℓ, columns: outaged line k; LODF_kk = −1). Verify it: for lesson 4's N-0 dispatch and every outage k, the predicted flows f_ℓ + LODF_ℓk · f_k must equal the flows recomputed with `ptdf(LINES.drop(k))`.

In [2]:
def lodf() -> pd.DataFrame:
    """LODF[l, k]: flow change on line l per MW that line k carried before its outage."""
    ### BEGIN SOLUTION
    out = pd.DataFrame(0.0, index=LINES.index, columns=LINES.index)
    for k in LINES.index:
        transfer = PTDF[LINES.bus0[k]] - PTDF[LINES.bus1[k]]
        out[k] = transfer / (1 - transfer[k])
        out.loc[k, k] = -1.0
    return out
    ### END SOLUTION

In [3]:
LODF = lodf()
f0 = flows(N0_DISPATCH)
for k in LINES.index:
    without_k = LINES.drop(k)
    recomputed = flows(N0_DISPATCH, ptdf(without_k))
    assert np.allclose(f0.drop(k) + LODF.loc[without_k.index, k] * f0[k], recomputed, atol=1e-6), k
assert np.isclose(secure_dispatch(LODF)["cost"], 107913.8, atol=0.1), "the lecture's SCOPF cost"
LODF.round(3)

                Zagreb-Rijeka  Rijeka-Velebit  Velebit-Split  Zagreb-Split  \
Zagreb-Rijeka          -1.000          -1.000         -1.000         0.592   
Rijeka-Velebit         -1.000          -1.000         -1.000         0.592   
Velebit-Split          -1.000          -1.000         -1.000         0.592   
Zagreb-Split            0.644           0.644          0.644        -1.000   
Zagreb-Osijek           0.356           0.356          0.356         0.408   
Osijek-Split            0.356           0.356          0.356         0.408   

                Zagreb-Osijek  Osijek-Split  
Zagreb-Rijeka           0.444         0.444  
Rijeka-Velebit          0.444         0.444  
Velebit-Split           0.444         0.444  
Zagreb-Split            0.556         0.556  
Zagreb-Osijek          -1.000        -1.000  
Osijek-Split           -1.000        -1.000  

**2. The full screening matrix**

Write `screening(dispatch, post_limits)` returning a table: one row per outage, one column per monitored line, entries the post-outage loading in % of `post_limits` (empty for the outaged line itself). Count the violations (> 100 %) for lesson 4's N-0 dispatch.

In [4]:
def screening(dispatch: pd.Series, post_limits=None) -> pd.DataFrame:
    """Post-outage loading (%) of every line for every single line outage."""
    ### BEGIN SOLUTION
    post_limits = pd.Series(LIMIT, index=LINES.index) if post_limits is None else post_limits
    f = flows(dispatch)
    rows = {}
    for k in LINES.index:
        after = (f + LODF[k] * f[k]).abs() / post_limits * 100
        after[k] = np.nan
        rows[f"{k} out"] = after
    return pd.DataFrame(rows).T
    ### END SOLUTION

In [5]:
table = screening(N0_DISPATCH)
violations = int((table > 100 + 1e-6).sum().sum())
assert violations == 12, "every outage overloads two lines"
print("violations:", violations)
table.round(0)

violations: 12


                    Zagreb-Rijeka  Rijeka-Velebit  Velebit-Split  \
Zagreb-Rijeka out             NaN             0.0          121.0   
Rijeka-Velebit out            0.0             NaN          121.0   
Velebit-Split out           121.0           121.0            NaN   
Zagreb-Split out            138.0           138.0           17.0   
Zagreb-Osijek out           125.0           125.0            4.0   
Osijek-Split out            109.0           109.0           13.0   

                    Zagreb-Split  Zagreb-Osijek  Osijek-Split  
Zagreb-Rijeka out          129.0           92.0          55.0  
Rijeka-Velebit out         129.0           92.0          55.0  
Velebit-Split out           51.0           49.0          12.0  
Zagreb-Split out             NaN           83.0          46.0  
Zagreb-Osijek out           96.0            NaN          38.0  
Osijek-Split out            75.0           38.0           NaN  

### Core

**3. Short-term emergency ratings**

Lines may carry more than their continuous rating for the minutes an operator needs to act after an outage. Let the post-outage limit be 100 %, 110 %, 120 %, 130 % and 140 % of 400 MW (the pre-outage limit stays 400 MW). Report the cost of the security-constrained dispatch and its excess over the N-0 cost (81,956.3 €/h). Write `emergency_ratings(factors)`.

In [6]:
def emergency_ratings(factors=(1.0, 1.1, 1.2, 1.3, 1.4)) -> pd.DataFrame:
    """Cost of N-1 security for each post-outage rating factor."""
    ### BEGIN SOLUTION
    n0 = secure_dispatch(LODF, security=False)["cost"]
    rows = {}
    for factor in factors:
        r = secure_dispatch(LODF, post_limits=pd.Series(LIMIT * factor, index=LINES.index))
        rows[factor] = {"post_limit_MW": LIMIT * factor, "cost_EUR_per_h": r["cost"], "cost_of_security": r["cost"] - n0,
                        "wind_MW": r["dispatch"]["wind Velebit"], "import_MW": r["dispatch"]["import Zagreb"]}
    return pd.DataFrame(rows).T.rename_axis("rating factor")
    ### END SOLUTION

In [7]:
ratings = emergency_ratings()
assert ratings.cost_of_security.is_monotonic_decreasing
assert np.isclose(ratings.loc[1.0, "cost_of_security"], 25957.5, atol=0.5)
ratings.round(1)

               post_limit_MW  cost_EUR_per_h  cost_of_security  wind_MW  \
rating factor                                                             
1.0                    400.0        107913.8           25957.5    400.0   
1.1                    440.0        101058.6           19102.3    440.0   
1.2                    480.0         94203.4           12247.1    480.0   
1.3                    520.0         87348.3            5392.0    520.0   
1.4                    560.0         81956.3               0.0    485.0   

               import_MW  
rating factor             
1.0                249.1  
1.1                192.6  
1.2                136.0  
1.3                 79.5  
1.4                 15.0  

**4. Interpretation**

How fast does the cost of security fall with the emergency rating, and from which rating is N-1 security free here? Why? What does an operator need, besides a higher rating, to rely on it?

<!-- BEGIN SOLUTION -->
The cost of security falls almost linearly with the emergency rating — by about 6,855 €/h for every 10 % (40 MW) — from 25,958 €/h at 100 % to 5,392 €/h at 130 % and nothing at 140 %. Each extra 40 MW of rating lets 40 MW more wind at Velebit run and replaces part of the imports at Zagreb. Security becomes free exactly when the emergency rating covers the worst post-outage flow of the N-0 dispatch: 552.4 MW after the loss of Zagreb–Split, or 138.1 % of 400 MW (re-solving at 138.1 % still costs 2 €/h, at 138.2 % nothing). From there on the N-0 dispatch is already N-1 secure in the emergency sense.

A higher rating is only half of the arrangement. It lets a line carry more **for minutes**, not for good, so the operator must be able to bring the flow back below the continuous rating within that time: fast redispatch from plants that can actually move, topology switching, or automatic protection schemes that trip generation — the remedial actions that make the dispatch **correctively** rather than preventively secure. It also needs reliable ratings (often measured: dynamic line rating) and a plan for what happens if the remedial action fails. The money saved is real, but it is paid for with less margin and more dependence on fast actions working.
<!-- END SOLUTION -->

### Challenge

**5. Which reinforcement is worth most?**

Upgrade **one** line at a time from 400 to 600 MW (both before and after an outage; all others stay at 400 MW). For each, report the cost saving of the N-0 dispatch and of the N-1 dispatch, against their base costs. Write `reinforcements()` and rank the lines by each saving.

In [8]:
def reinforcements(upgrade_mw: float = 600.0) -> pd.DataFrame:
    """Cost saving (EUR/h) of upgrading each line, without and with N-1 security."""
    ### BEGIN SOLUTION
    base0 = secure_dispatch(LODF, security=False)["cost"]
    base1 = secure_dispatch(LODF)["cost"]
    rows = {}
    for line in LINES.index:
        limits = pd.Series(LIMIT, index=LINES.index)
        limits[line] = upgrade_mw
        rows[line] = {"saving_N-0": base0 - secure_dispatch(LODF, limits=limits, security=False)["cost"],
                      "saving_N-1": base1 - secure_dispatch(LODF, limits=limits)["cost"]}
    out = pd.DataFrame(rows).T
    out["rank_N-0"] = out["saving_N-0"].rank(ascending=False, method="min")
    out["rank_N-1"] = out["saving_N-1"].rank(ascending=False, method="min")
    return out
    ### END SOLUTION

In [9]:
upgrades = reinforcements()
assert (upgrades[["saving_N-0", "saving_N-1"]] >= -1e-6).all().all()
upgrades.round(1)

                saving_N-0  saving_N-1  rank_N-0  rank_N-1
Zagreb-Rijeka       8437.5      2206.9       2.0       2.0
Rijeka-Velebit     10956.3      4011.4       1.0       1.0
Velebit-Split          0.0         0.0       3.0       3.0
Zagreb-Split           0.0         0.0       3.0       3.0
Zagreb-Osijek          0.0         0.0       3.0       3.0
Osijek-Split           0.0         0.0       3.0       3.0

**6. Interpretation**

Does N-1 security change which reinforcement is worth most? Explain with the lecture's corridors and LODFs. What does that mean for grid planning studies that check only N-0?

<!-- BEGIN SOLUTION -->
No — the ranking stays the same: upgrading Rijeka–Velebit is worth most and Zagreb–Rijeka second; the other four lines are worth nothing on their own. But N-1 shrinks the values a lot: 4,011 instead of 10,956 €/h for Rijeka–Velebit, 2,207 instead of 8,438 €/h for Zagreb–Rijeka. The lecture's LODFs explain it. Under N-1 the binding constraints belong to the whole Velebit–Rijeka–Zagreb **corridor** — Zagreb–Rijeka, Rijeka–Velebit and Velebit–Split fail together and share each other's flow after an outage — so upgrading one link leaves its partners binding. Upgrading the whole corridor tells the story: Zagreb–Rijeka and Rijeka–Velebit together save 5,517 €/h under N-1, and all three lines 11,517 €/h — more than the best single line was worth even under N-0.

For planning, an N-0 study gets the direction right here but the size wrong: it would overstate the value of a single line upgrade by a factor of about 3 and miss that under the security rules operators actually apply, value comes from reinforcing corridors, not lines. That is why grid planning studies check reinforcements against contingencies, not on the intact network alone.
<!-- END SOLUTION -->